# Part A — CNN Image Classification (Fast Version)
### Simple CNN vs ResNet-18 on CIFAR-10

In [ ]:
# CELL 1 — Install libraries
import sys
!{sys.executable} -m pip install torch torchvision numpy matplotlib seaborn scikit-learn --quiet
print('✅ Done!')

In [ ]:
# CELL 2 — Imports & Settings
import os, time, torch, torchvision
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import confusion_matrix

torch.manual_seed(42)

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS     = 5          # ✅ Only 5 epochs — runs in ~5 mins on CPU
BATCH      = 128        # ✅ Larger batch = faster training
LR         = 0.001
CLASSES    = ['airplane','automobile','bird','cat','deer',
              'dog','frog','horse','ship','truck']

os.makedirs('./outputs', exist_ok=True)
print(f'✅ Device: {DEVICE} | Epochs: {EPOCHS} | Batch: {BATCH}')

In [ ]:
# CELL 3 — Load CIFAR-10 (uses only 10,000 samples for speed)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

# Download full dataset
full_train = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform)
full_test  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# ✅ Use only 8000 train + 2000 val + 2000 test for faster runs
train_data, val_data, _ = random_split(full_train, [8000, 2000, 40000])
test_data,  _           = random_split(full_test,  [2000, 8000])

train_loader = DataLoader(train_data, batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=BATCH, shuffle=False)
test_loader  = DataLoader(test_data,  batch_size=BATCH, shuffle=False)

print(f'✅ Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}')

In [ ]:
# CELL 4 — Simple CNN Model
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, 3, padding=1),  # Conv
            nn.BatchNorm2d(32),              # BatchNorm
            nn.ReLU(),                       # ReLU
            nn.MaxPool2d(2),                 # Pool → 16x16
            # Block 2
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),                 # → 8x8
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(64*8*8, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

model_cnn = SimpleCNN().to(DEVICE)
params_cnn = sum(p.numel() for p in model_cnn.parameters())
print(f'✅ Simple CNN ready | Parameters: {params_cnn:,}')

In [ ]:
# CELL 5 — ResNet-18 Transfer Learning Model
model_resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model_resnet.fc = nn.Linear(512, 10)   # replace final layer for 10 classes
model_resnet = model_resnet.to(DEVICE)

params_resnet = sum(p.numel() for p in model_resnet.parameters())
print(f'✅ ResNet-18 ready  | Parameters: {params_resnet:,}')

In [ ]:
# CELL 6 — Training Function
def train(model, name):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)

    train_losses, val_losses = [], []
    train_accs,   val_accs   = [], []
    start = time.time()

    print(f'\n🚀 Training {name}...')
    for epoch in range(1, EPOCHS+1):

        # — Train —
        model.train()
        loss_sum, correct, total = 0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item()
            correct  += out.argmax(1).eq(labels).sum().item()
            total    += labels.size(0)
        train_losses.append(loss_sum / len(train_loader))
        train_accs.append(100 * correct / total)

        # — Validate —
        model.eval()
        loss_sum, correct, total = 0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                out  = model(imgs)
                loss = criterion(out, labels)
                loss_sum += loss.item()
                correct  += out.argmax(1).eq(labels).sum().item()
                total    += labels.size(0)
        val_losses.append(loss_sum / len(val_loader))
        val_accs.append(100 * correct / total)

        print(f'  Epoch {epoch}/{EPOCHS} | Train Acc: {train_accs[-1]:.1f}% | Val Acc: {val_accs[-1]:.1f}%')

    elapsed = time.time() - start
    print(f'⏱ Done in {elapsed/60:.1f} min | Best Val Acc: {max(val_accs):.1f}%')
    return train_losses, val_losses, train_accs, val_accs, elapsed

print('✅ Training function ready!')

In [ ]:
# CELL 7 — Train Simple CNN  (~2-3 mins)
tl_cnn, vl_cnn, ta_cnn, va_cnn, time_cnn = train(model_cnn, 'Simple CNN')

In [ ]:
# CELL 8 — Train ResNet-18  (~3-4 mins)
tl_res, vl_res, ta_res, va_res, time_res = train(model_resnet, 'ResNet-18')

In [ ]:
# CELL 9 — Plot Training Curves
epochs = range(1, EPOCHS+1)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('CNN vs ResNet-18 — Training Curves', fontsize=15, fontweight='bold')

# Simple CNN Loss
axes[0,0].plot(epochs, tl_cnn, label='Train', color='steelblue')
axes[0,0].plot(epochs, vl_cnn, label='Val',   color='steelblue', linestyle='--')
axes[0,0].set_title('Simple CNN — Loss');  axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

# Simple CNN Accuracy
axes[0,1].plot(epochs, ta_cnn, label='Train', color='steelblue')
axes[0,1].plot(epochs, va_cnn, label='Val',   color='steelblue', linestyle='--')
axes[0,1].set_title('Simple CNN — Accuracy'); axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

# ResNet Loss
axes[1,0].plot(epochs, tl_res, label='Train', color='darkorange')
axes[1,0].plot(epochs, vl_res, label='Val',   color='darkorange', linestyle='--')
axes[1,0].set_title('ResNet-18 — Loss');      axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

# ResNet Accuracy
axes[1,1].plot(epochs, ta_res, label='Train', color='darkorange')
axes[1,1].plot(epochs, va_res, label='Val',   color='darkorange', linestyle='--')
axes[1,1].set_title('ResNet-18 — Accuracy');  axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('./outputs/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: training_curves.png')

In [ ]:
# CELL 10 — Test Accuracy + Confusion Matrix
def get_predictions(model):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(DEVICE)
            preds = model(imgs).argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    acc = 100 * sum(p==l for p,l in zip(all_preds,all_labels)) / len(all_labels)
    return np.array(all_preds), np.array(all_labels), acc

preds_cnn, labels_gt, acc_cnn     = get_predictions(model_cnn)
preds_res, labels_gt, acc_resnet  = get_predictions(model_resnet)

print(f'Simple CNN  Test Accuracy : {acc_cnn:.2f}%')
print(f'ResNet-18   Test Accuracy : {acc_resnet:.2f}%')

# Plot confusion matrices side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')

for ax, preds, title, cmap in [
    (ax1, preds_cnn, f'Simple CNN ({acc_cnn:.1f}%)',    'Blues'),
    (ax2, preds_res, f'ResNet-18  ({acc_resnet:.1f}%)', 'Oranges')
]:
    cm = confusion_matrix(labels_gt, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig('./outputs/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: confusion_matrices.png')

In [ ]:
# CELL 11 — Final Comparison Table
print('='*55)
print('         FINAL MODEL COMPARISON')
print('='*55)
print(f'{"Metric":<25} {"Simple CNN":>12} {"ResNet-18":>12}')
print('-'*55)
print(f'{"Test Accuracy":<25} {acc_cnn:>11.2f}% {acc_resnet:>11.2f}%')
print(f'{"Parameters":<25} {params_cnn:>12,} {params_resnet:>12,}')
print(f'{"Model Size (MB)":<25} {params_cnn*4/1024/1024:>11.2f}  {params_resnet*4/1024/1024:>11.2f}')
print(f'{"Training Time (min)":<25} {time_cnn/60:>11.1f}  {time_res/60:>11.1f}')
print(f'{"Best Val Accuracy":<25} {max(va_cnn):>11.1f}% {max(va_res):>11.1f}%')
print('='*55)
print('\n🎉 Part A Complete!')
print('📁 Outputs saved in ./outputs/')
print('   → training_curves.png')
print('   → confusion_matrices.png')